# Módulo 9 · Clase 5 — Entrenamiento DL

**Proyecto Final: Sistema Inteligente de Predicción y Exposición de Modelos**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Red neuronal con TensorFlow/Keras para datos tabulares |
| 2 | Clasificación binaria + regresión sobre vinos |
| 3 | Callbacks: EarlyStopping, ModelCheckpoint |
| 4 | Comparativa ML vs DL |
| 5 | Guardar modelo en formato `.keras` |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, mean_absolute_error
import joblib

print(f"TF version: {tf.__version__}")

# Reproducibilidad
tf.random.set_seed(42)
np.random.seed(42)

df = pd.read_parquet('data/processed/vinos_procesado.parquet')
print(f"Dataset: {df.shape}")

---
## 1. Preparar datos

In [ ]:
FEATURES = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol',
    'so2_ratio', 'acid_index', 'density_diff', 'tipo_encoded'
]

X = df[FEATURES].values.astype(np.float32)
y_clf = df['buen_vino'].values.astype(np.float32)     # clasificación binaria
y_reg = df['quality'].values.astype(np.float32)       # regresión

# Split
X_tr, X_te, yc_tr, yc_te, yr_tr, yr_te = train_test_split(
    X, y_clf, y_reg, test_size=0.2, random_state=42, stratify=y_clf
)
X_tr, X_val, yc_tr, yc_val, yr_tr, yr_val = train_test_split(
    X_tr, yc_tr, yr_tr, test_size=0.15, random_state=42
)

# Normalización
scaler = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_tr).astype(np.float32)
X_val_sc = scaler.transform(X_val).astype(np.float32)
X_te_sc  = scaler.transform(X_te).astype(np.float32)

# Guardar scaler para la API
joblib.dump(scaler, 'models/dl_scaler.joblib')

print(f"Train: {X_tr_sc.shape} | Val: {X_val_sc.shape} | Test: {X_te_sc.shape}")
print(f"Clase 1 en train: {yc_tr.mean():.2%}")

---
## 2. Modelo de clasificación binaria

In [ ]:
def crear_modelo_clf(input_dim, dropout=0.3, l2=0.001):
    """Red densa para clasificación binaria de calidad de vino."""
    inp = keras.Input(shape=(input_dim,), name='input')
    x = layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(l2))(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid', name='output')(x)
    return keras.Model(inp, out, name='vino_clf')

modelo_clf = crear_modelo_clf(input_dim=X_tr_sc.shape[1])
modelo_clf.summary()

modelo_clf.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

In [ ]:
Path('logs/clf').mkdir(parents=True, exist_ok=True)

clf_callbacks = [
    callbacks.EarlyStopping(monitor='val_auc', patience=15,
                            mode='max', restore_best_weights=True),
    callbacks.ModelCheckpoint('models/best_clf.keras', monitor='val_auc',
                               mode='max', save_best_only=True, verbose=0),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                 patience=7, min_lr=1e-6),
]

hist_clf = modelo_clf.fit(
    X_tr_sc, yc_tr,
    validation_data=(X_val_sc, yc_val),
    epochs=200, batch_size=64,
    callbacks=clf_callbacks,
    verbose=0
)

print(f"Entrenamiento terminado en época {len(hist_clf.history['loss'])}")

# Curvas de aprendizaje
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metrica in zip(axes, ['loss', 'auc']):
    ax.plot(hist_clf.history[metrica], label='train')
    ax.plot(hist_clf.history[f'val_{metrica}'], label='val')
    ax.set_title(metrica); ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Curvas de aprendizaje — Clasificación')
plt.tight_layout()
plt.show()

In [ ]:
# Evaluación en test
y_prob_dl = modelo_clf.predict(X_te_sc, verbose=0).ravel()
y_pred_dl = (y_prob_dl >= 0.5).astype(int)

f1_dl  = f1_score(yc_te, y_pred_dl)
auc_dl = roc_auc_score(yc_te, y_prob_dl)
acc_dl = (y_pred_dl == yc_te).mean()

print(f"Red neuronal — Test F1: {f1_dl:.4f}  AUC: {auc_dl:.4f}  Acc: {acc_dl:.4f}")

# Comparar con Random Forest de C4
try:
    rf_pipe = joblib.load('models/rf_vinos_v1.joblib')
    y_pred_rf = rf_pipe.predict(X_te)
    y_prob_rf = rf_pipe.predict_proba(X_te)[:, 1]
    f1_rf  = f1_score(yc_te, y_pred_rf)
    auc_rf = roc_auc_score(yc_te, y_prob_rf)
    print(f"Random Forest    — Test F1: {f1_rf:.4f}  AUC: {auc_rf:.4f}  Acc: {(y_pred_rf==yc_te).mean():.4f}")
except FileNotFoundError:
    print("(Modelo RF no encontrado — ejecuta primero M9_C4)")

---
## 3. Modelo de regresión: predecir puntuación exacta

In [ ]:
def crear_modelo_reg(input_dim, dropout=0.2):
    """Red densa para regresión de puntuación de vino (0-10)."""
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(128, activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation='relu')(x)
    # Sin activación final = regresión libre
    out = layers.Dense(1, name='quality_pred')(x)
    return keras.Model(inp, out, name='vino_reg')

modelo_reg = crear_modelo_reg(input_dim=X_tr_sc.shape[1])
modelo_reg.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss='mse',
    metrics=['mae']
)

reg_callbacks = [
    callbacks.EarlyStopping(monitor='val_mae', patience=20,
                            restore_best_weights=True),
    callbacks.ModelCheckpoint('models/best_reg.keras', monitor='val_mae',
                               save_best_only=True, verbose=0),
]

hist_reg = modelo_reg.fit(
    X_tr_sc, yr_tr,
    validation_data=(X_val_sc, yr_val),
    epochs=200, batch_size=64,
    callbacks=reg_callbacks, verbose=0
)

print(f"Regresión terminada en época {len(hist_reg.history['loss'])}")
y_pred_reg = modelo_reg.predict(X_te_sc, verbose=0).ravel()
mae = mean_absolute_error(yr_te, y_pred_reg)
rmse = np.sqrt(np.mean((yr_te - y_pred_reg)**2))
print(f"Test MAE: {mae:.4f}  RMSE: {rmse:.4f}")

In [ ]:
# Scatter: predicciones vs reales
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter
ax = axes[0]
jitter = np.random.normal(0, 0.1, len(yr_te))
ax.scatter(yr_te + jitter, y_pred_reg, alpha=0.3, s=10, color='steelblue')
mn, mx = min(yr_te.min(), y_pred_reg.min()), max(yr_te.max(), y_pred_reg.max())
ax.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Predicción perfecta')
ax.set_xlabel('Quality real'); ax.set_ylabel('Quality predicha')
ax.set_title(f'Regresión (MAE={mae:.3f})')
ax.legend(); ax.grid(True, alpha=0.3)

# Distribución de errores
ax = axes[1]
errores = y_pred_reg - yr_te
ax.hist(errores, bins=40, color='coral', edgecolor='white')
ax.axvline(0, color='black', lw=2)
ax.set_xlabel('Error (pred - real)'); ax.set_ylabel('Frecuencia')
ax.set_title(f'Distribución de errores (sesgo={errores.mean():.3f})')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Guardar modelos y metadatos

In [ ]:
from datetime import datetime

# Guardar modelos en formato .keras
modelo_clf.save('models/dl_clf_v1.keras')
modelo_reg.save('models/dl_reg_v1.keras')
print("✓ Modelos guardados")

# Verificar carga
m_test = keras.models.load_model('models/dl_clf_v1.keras')
pred_test = m_test.predict(X_te_sc[:3], verbose=0)
print(f"  Verificación carga: {pred_test.ravel().round(3)}")

# Metadatos
meta_dl = {
    'nombre': 'DL Wine Quality Classifier + Regressor',
    'fecha': datetime.now().isoformat(),
    'features': FEATURES,
    'n_train': int(X_tr_sc.shape[0]),
    'n_test': int(X_te_sc.shape[0]),
    'clf': {
        'archivo': 'models/dl_clf_v1.keras',
        'scaler':  'models/dl_scaler.joblib',
        'f1':  round(float(f1_dl), 4),
        'auc': round(float(auc_dl), 4),
        'acc': round(float(acc_dl), 4),
        'epocas_entrenadas': len(hist_clf.history['loss'])
    },
    'reg': {
        'archivo': 'models/dl_reg_v1.keras',
        'mae':  round(float(mae), 4),
        'rmse': round(float(rmse), 4),
        'epocas_entrenadas': len(hist_reg.history['loss'])
    }
}

with open('models/dl_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta_dl, f, indent=2, ensure_ascii=False)

print("✓ Metadatos guardados en models/dl_meta.json")

# Listar archivos del directorio models/
print("\nContenido de models/:")
for p in sorted(Path('models').iterdir()):
    print(f"  {p.name:40} {p.stat().st_size/1024:.0f} KB")

---
## Práctica

### Ejercicio 1 — Comparativa de arquitecturas

**Enunciado:** Prueba tres profundidades de red y compara el AUC en validación:

1. **Shallow**: 1 capa Dense(64)
2. **Medium**: 2 capas Dense(128, 64)
3. **Deep**: 4 capas Dense(256, 128, 64, 32)

Usa EarlyStopping(patience=10) en los tres. ¿Cuál generaliza mejor?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

arquitecturas = {
    'Shallow': [64],
    'Medium':  [128, 64],
    'Deep':    [256, 128, 64, 32],
}

for nombre, dims in arquitecturas.items():
    inp = keras.Input(shape=(X_tr_sc.shape[1],))
    x = inp
    for d in dims:
        x = layers.Dense(d, activation='relu')(x)
        x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = keras.Model(inp, out)
    m.compile(optimizer='adam', loss='binary_crossentropy',
              metrics=[keras.metrics.AUC(name='auc')])

    h = m.fit(X_tr_sc, yc_tr,
              validation_data=(X_val_sc, yc_val),
              epochs=100, batch_size=64, verbose=0,
              callbacks=[callbacks.EarlyStopping(monitor='val_auc', patience=10,
                                                  mode='max', restore_best_weights=True)])

    best_auc = max(h.history['val_auc'])
    epocas   = len(h.history['loss'])
    print(f"{nombre:10}: val_AUC={best_auc:.4f}  épocas={epocas}  params={m.count_params():,}")
```
</details>

---
### Ejercicio 2 — Modelo multitarea

**Enunciado:** Crea un modelo que prediga SIMULTÁNEAMENTE la clasificación binaria y la puntuación exacta (Functional API con dos salidas):

```python
# Dos salidas
out_clf = layers.Dense(1, activation='sigmoid', name='buen_vino')(x)
out_reg = layers.Dense(1, name='quality')(x)
```

Combina las pérdidas: `loss={'buen_vino': 'bce', 'quality': 'mse'}`

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
from tensorflow import keras
from tensorflow.keras import layers

inp = keras.Input(shape=(X_tr_sc.shape[1],), name='features')
x = layers.Dense(128, activation='relu')(inp)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(32, activation='relu')(x)

out_clf = layers.Dense(1, activation='sigmoid', name='buen_vino')(x)
out_reg = layers.Dense(1, name='quality')(x)

modelo_mt = keras.Model(inp, [out_clf, out_reg], name='multitarea')
modelo_mt.compile(
    optimizer='adam',
    loss={'buen_vino': 'binary_crossentropy', 'quality': 'mse'},
    loss_weights={'buen_vino': 1.0, 'quality': 0.5},  # ponderar contribuciones
    metrics={'buen_vino': keras.metrics.AUC(name='auc'), 'quality': 'mae'}
)

h = modelo_mt.fit(
    X_tr_sc,
    {'buen_vino': yc_tr, 'quality': yr_tr},
    validation_data=(X_val_sc, {'buen_vino': yc_val, 'quality': yr_val}),
    epochs=100, batch_size=64, verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)]
)

pred_clf, pred_reg = modelo_mt.predict(X_te_sc, verbose=0)
from sklearn.metrics import roc_auc_score, mean_absolute_error
print(f"AUC clasificación: {roc_auc_score(yc_te, pred_clf.ravel()):.4f}")
print(f"MAE regresión:     {mean_absolute_error(yr_te, pred_reg.ravel()):.4f}")
```
</details>

---
### Ejercicio 3 — Función `predecir_vino_dl()`

**Enunciado:** Crea una función equivalente a la de C4 pero usando el modelo DL:

```python
def predecir_vino_dl(datos: dict) -> dict:
    # Devuelve clasificación, probabilidad Y puntuación numérica
    return {
        'prediccion': 'bueno',
        'probabilidad': 0.82,
        'puntuacion_estimada': 7.3,
        'modelo': 'dl_v1'
    }
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import joblib
import numpy as np
from tensorflow import keras

CAMPOS_REQ = [
    'fixed acidity','volatile acidity','citric acid','residual sugar',
    'chlorides','free sulfur dioxide','total sulfur dioxide',
    'density','pH','sulphates','alcohol','tipo'
]
FEATURES_ORDEN = [
    'fixed acidity','volatile acidity','citric acid','residual sugar',
    'chlorides','free sulfur dioxide','total sulfur dioxide','density',
    'pH','sulphates','alcohol','so2_ratio','acid_index','density_diff','tipo_encoded'
]

_cache_clf = _cache_reg = _cache_sc = None

def predecir_vino_dl(datos: dict, umbral: float = 0.5) -> dict:
    global _cache_clf, _cache_reg, _cache_sc
    faltantes = [c for c in CAMPOS_REQ if c not in datos]
    if faltantes:
        raise ValueError(f"Campos faltantes: {faltantes}")

    if _cache_clf is None:
        _cache_clf = keras.models.load_model('models/dl_clf_v1.keras')
        _cache_reg = keras.models.load_model('models/dl_reg_v1.keras')
        _cache_sc  = joblib.load('models/dl_scaler.joblib')

    d = dict(datos)
    d['so2_ratio']    = d['free sulfur dioxide'] / (d['total sulfur dioxide'] + 1e-8)
    d['acid_index']   = d['fixed acidity'] - d['volatile acidity']
    d['density_diff'] = abs(d['density'] - 1.0)
    d['tipo_encoded'] = 1 if d['tipo'] == 'tinto' else 0

    X = _cache_sc.transform([[d[f] for f in FEATURES_ORDEN]])
    prob = float(_cache_clf.predict(X, verbose=0)[0, 0])
    punt = float(_cache_reg.predict(X, verbose=0)[0, 0])

    return {
        'prediccion':          'bueno' if prob >= umbral else 'normal',
        'probabilidad':        round(prob, 4),
        'puntuacion_estimada': round(punt, 2),
        'modelo': 'dl_v1'
    }

print(predecir_vino_dl({
    'fixed acidity': 7.4, 'volatile acidity': 0.7, 'citric acid': 0.0,
    'residual sugar': 1.9, 'chlorides': 0.076, 'free sulfur dioxide': 11.0,
    'total sulfur dioxide': 34.0, 'density': 0.9978, 'pH': 3.51,
    'sulphates': 0.56, 'alcohol': 9.4, 'tipo': 'tinto'
}))
```
</details>

---
## Resumen

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Modelo tabular
inp = keras.Input(shape=(n_features,))
x   = layers.Dense(128, activation='relu')(inp)
x   = layers.BatchNormalization()(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(1, activation='sigmoid')(x)  # clasificación
# out = layers.Dense(1)(x)                       # regresión

modelo = keras.Model(inp, out)
modelo.compile(optimizer='adam', loss='binary_crossentropy',
               metrics=[keras.metrics.AUC()])

# Callbacks esenciales
cb = [
    callbacks.EarlyStopping(monitor='val_auc', patience=15, restore_best_weights=True),
    callbacks.ModelCheckpoint('mejor.keras', monitor='val_auc', save_best_only=True),
]
modelo.fit(X_tr, y_tr, validation_data=(X_val, y_val),
           epochs=200, batch_size=64, callbacks=cb)

# Guardar
modelo.save('models/modelo.keras')
modelo_cargado = keras.models.load_model('models/modelo.keras')
```

## ¿Qué viene en la próxima clase?
En la **Clase 6** expondremos los modelos entrenados a través de una **API REST** con FastAPI y Pydantic.